In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\saksh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

In [4]:
# Load datasets
train_df = pd.read_csv("../data/twitter_training.csv")
val_df   = pd.read_csv("../data/twitter_validation.csv")

train_df.columns = ['tweet_id', 'entity', 'sentiment', 'tweet_content']
val_df.columns   = ['tweet_id', 'entity', 'sentiment', 'tweet_content']

train_df['tweet_content'] = train_df['tweet_content'].apply(clean_text)
val_df['tweet_content']   = val_df['tweet_content'].apply(clean_text)

In [5]:
print(train_df['tweet_content'].isna().sum())
print(val_df['tweet_content'].isna().sum())

0
0


In [6]:
train_df = train_df.dropna(subset=['tweet_content'])
val_df   = val_df.dropna(subset=['tweet_content'])

In [7]:
train_df['tweet_content'] = train_df['tweet_content'].astype(str).fillna("")
val_df['tweet_content']   = val_df['tweet_content'].astype(str).fillna("")

train_df = train_df[train_df['tweet_content'].str.strip() != ""]
val_df   = val_df[val_df['tweet_content'].str.strip() != ""]

In [8]:
# Filter only allowed classes
allowed = ['Negative', 'Neutral', 'Positive']
train_df = train_df[train_df['sentiment'].isin(allowed)]
val_df   = val_df[val_df['sentiment'].isin(allowed)]

# Label encoding
label_map = {'Negative':0, 'Neutral':1, 'Positive':2}
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label']   = val_df['sentiment'].map(label_map)

train_df.to_csv("../data/clean_train.csv", index=False)
val_df.to_csv("../data/clean_val.csv", index=False)
print("Preprocessing completed!")

Preprocessing completed!
